In [28]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os, subprocess, json
from pathlib import Path
from datetime import datetime

load_dotenv(override=True)

True

In [29]:
# ── Paths ──────────────────────────────────────────────────────────────────
NPM_ROOT = subprocess.check_output(["npm", "root", "-g"]).decode().strip()
TAVILY_PATH  = os.path.join(NPM_ROOT, "tavily-mcp", "build", "index.js")

# ── Server params ──────────────────────────────────────────────────────────

# LOẠI 1: Local hoàn toàn — Memory (knowledge graph, RAM)
#os.makedirs("./memory", exist_ok=True)

MEMORY_PARAMS = {
    "command": "npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {
        "LIBSQL_URL": "file:./memory/bds.db"
    }
}

# LOẠI 2: Local server, gọi API ngoài — Dữ liệu tài chính VN
PROJECT_DIR   = Path.cwd()          # folder where your notebook lives
SERVER_FILE   = PROJECT_DIR / "dulieu_vn_server.py"

DULIEU_VN_PARAMS = {
    "command": "python",
    "args":    [str(SERVER_FILE)],  
}

# LOẠI 3: Remote server qua HTTPS — Tavily Search
TAVILY_PARAMS = {
    "command": "node",
    "args":    [TAVILY_PATH],
    "env":     {**os.environ, "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY", "")}
}

In [3]:
# Memory server
instructions_mem = """
Bạn là trợ lý phân tích BĐS có trí nhớ dài hạn qua Knowledge Graph.

QUY TRÌNH XỬ LÝ BẮT BUỘC (PHẢI LÀM THEO THỨ TỰ):

Bước 1: Kiểm tra danh tính. 
- Ngay khi nhận câu hỏi, LUÔN LUÔN gọi tool `search_nodes` với từ khóa là tên khách hàng hoặc các thông tin liên quan.
- KHÔNG ĐƯỢC trả lời ngay nếu chưa có kết quả từ memory.

Bước 2: Phân tích thông tin.
- Nếu `search_nodes` trả về dữ liệu: Sử dụng dữ liệu đó để tư vấn.
- Nếu không thấy dữ liệu: Lúc đó mới được phép hỏi lại người dùng.

Bước 3: Cập nhật thông tin.
- Nếu người dùng cung cấp thêm thông tin mới (tuổi, ngân sách, sở thích...), dùng `create_entities` và `add_observations` để lưu lại ngay lập tức.

Ghi nhớ: Bạn không bao giờ được nói "Tôi không biết" hoặc "Vui lòng cung cấp thông tin" nếu bạn chưa thực hiện bước tìm kiếm trong memory.
"""

async with MCPServerStdio(params=MEMORY_PARAMS, client_session_timeout_seconds=15) as mcp_mem:
    agent_mem = Agent(
        name="tro-ly-bds-co-nho",
        instructions=instructions_mem,
        model="gpt-4o-mini",
        mcp_servers=[mcp_mem]
    )
    with trace("memory-demo-request1"):
        r1 = await Runner.run(
            agent_mem,
            "Xin chào! Tôi là Thanh Tâm, 30 tuổi, ngân sách 4 tỷ VNĐ, "
            "muốn mua chung cư tại Quận 7 hoặc Bình Thạnh HCM để đầu tư cho thuê. "
            "Tôi ưu tiên gần trường quốc tế.",
            max_turns=8
        )
    print(f"Request 1:")
    print(r1.final_output)
    print()

    with trace("memory-demo-request2"):
        r2 = await Runner.run(
            agent_mem,
            "Tôi là Thanh Tâm đây, dựa trên yêu cầu tôi vừa nêu, bạn có gợi ý gì không ?",
            max_turns=8
        )
    print(f"Request 2:")
    print(r2.final_output)


Request 1:
Hiện tại, tôi không tìm thấy thông tin cụ thể về các chung cư tại Quận 7 hoặc Bình Thạnh gần các trường quốc tế. Tuy nhiên, với ngân sách 4 tỷ VNĐ, bạn có thể tham khảo một số dự án chung cư sau:

1. **Quận 7**:
   - Có một số dự án nổi bật như Phú Mỹ Hưng, nơi có nhiều trường quốc tế và cơ sở hạ tầng phát triển.

2. **Bình Thạnh**:
   - Khu vực này có chung cư gần các trường học nổi tiếng và tiện ích đầy đủ, bạn có thể tham khảo những dự án gần bến xe bus hoặc metro.

Nếu bạn cần thêm thông tin chi tiết về từng dự án hoặc các tiêu chí cụ thể khác, hãy cho tôi biết!

Request 2:
Chào bạn, Thanh Tâm! Dựa trên thông tin mà tôi đã có, bạn 30 tuổi, có ngân sách 4 tỷ VNĐ, đang tìm mua chung cư tại Quận 7 hoặc Bình Thạnh, TP.HCM, với ý định đầu tư cho thuê và ưu tiên gần trường quốc tế.

Dưới đây là một số gợi ý cho bạn:

1. **Quận 7**:
   - **Phú Mỹ Hưng**: Khu vực này nổi bật với nhiều dự án chung cư cao cấp, gần các trường quốc tế như Apollo và Saigon South International School.

In [3]:
# 2. Server local nhung goi Vietcombank + SJC APIs
async with MCPServerStdio(params=DULIEU_VN_PARAMS, client_session_timeout_seconds=30) as mcp_vn:
    tools = await mcp_vn.list_tools()
    print(f"Tools ({len(tools)}):")
    for t in tools:
        print(f"- {t.name}: {t.description.split(chr(10))[0]}")

Tools (7):
- ty_gia_ngoai_te: Lấy tỷ giá hối đoái của một loại ngoại tệ so với VNĐ từ Vietcombank.
- so_sanh_nhieu_ngoai_te: So sánh tỷ giá của nhiều loại ngoại tệ cùng lúc.
- gia_vang_hom_nay: Lấy giá vàng hiện tại từ SJC.
- tat_ca_loai_vang: Lấy giá vàng tất cả các loại (SJC miếng, nhẫn, PNJ, DOJI...).
- tim_nha_dat: Tìm kiếm bất động sản theo khu vực, loại, và giới hạn ngân sách.
- thong_ke_gia_thi_truong: Thống kê giá bất động sản trung bình, min, max theo từng khu vực.
- tinh_roi_dau_tu: Tính toán hiệu quả đầu tư cho thuê bất động sản (gross yield và thời gian hoàn vốn).


In [ ]:
# Hỏi thông tin tài chính
instructions_tc = """
Bạn là chuyên gia tư vấn tài chính cá nhân Việt Nam.
Cung cấp thông tin tỷ giá và giá vàng chính xác từ công cụ, không đoán mò.
Trả lời ngắn gọn, format số theo chuẩn VN (dấu phân cách nghìn).
"""
async with MCPServerStdio(params=DULIEU_VN_PARAMS, client_session_timeout_seconds=15) as mcp_vn:
    agent_tc = Agent(
        name="tu-van-tai-chinh",
        model="gpt-4o-mini",
        mcp_servers=[mcp_vn]
    )

    with trace('demo-tai-chinh-vn'):
        r = await Runner.run(
            agent_tc,
            f"Hôm nay {datetime.now().strftime('%d/%m/%Y')}, cho tôi biết:\n"
            "1. Tỷ giá USD, EUR, và JPY so với VNĐ\n"
            "2. Giá vàng SJC hiện tại\n"
            "3. Nếu tôi đổi 1.000 USD, nhận được bao nhiêu VNĐ?",
            max_turns=8
        )
    print(r.final_output)

Dưới đây là thông tin bạn yêu cầu:

### 1. Tỷ giá ngoại tệ so với VNĐ (08/04/2026)
- **USD (US Dollar)**:
  - Tỷ giá mua tiền mặt: 26,111.00 VNĐ
  - Tỷ giá mua chuyển khoản: 26,141.00 VNĐ
  - Tỷ giá bán: 26,361.00 VNĐ

- **EUR (Euro)**:
  - Tỷ giá mua tiền mặt: 29,939.33 VNĐ
  - Tỷ giá mua chuyển khoản: 30,241.75 VNĐ
  - Tỷ giá bán: 31,517.66 VNĐ

- **JPY (Yen)**:
  - Tỷ giá mua tiền mặt: 160.22 VNĐ
  - Tỷ giá mua chuyển khoản: 161.84 VNĐ
  - Tỷ giá bán: 170.40 VNĐ

### 2. Giá vàng SJC hiện tại
Không lấy được thông tin, nhưng bạn có thể kiểm tra trên trang web của các hãng vàng như SJC, DOJI hoặc PNJ.

### 3. Số VNĐ nhận được khi đổi 1.000 USD
Với tỷ giá bán 26,361.00 VNĐ cho 1 USD, khi bạn đổi 1.000 USD, bạn sẽ nhận được:
\[ 
1,000 \times 26,361.00 = 26,361,000 VNĐ 
\]

Nếu bạn có thêm câu hỏi nào khác, hãy cho tôi biết!


In [5]:
# Phân tích bds
from dulieu_vn import tim_bat_dong_san, thong_ke_thi_truong, tinh_roi_cho_thue

stats = thong_ke_thi_truong()
for s in stats:
    print(f" {s['khu_vuc']:<18} ({s['loai']:<10})" f"TB: {s['gia_trung_binh']:<25} | {s['so_tin_rao']} tin")

print("\nTop tin rao HCM dưới 4 tyr:")
for p in tim_bat_dong_san(khu_vuc="HCM", gia_max_ty=4.0):
    print(f" {p['khu_vuc']:<18} {p['dien_tich']}m2 - " f"{p['gia_m2']} - {p['gia_tong']}")

print("\nROI Quận 7 - căn 68m2 - 3.74 tỷ:")
roi = tinh_roi_cho_thue(3.74, 68, "Quận 7 HCM")
for k, v in roi.items():
    print(f" {k}: {v}")


 Toàn quốc          (Chung cư  )TB: 55tr/m2                   | 150 tin

Top tin rao HCM dưới 4 tyr:
 Quận 7 HCM         68.0m2 - 55000000.0 - 3740000000.0

ROI Quận 7 - căn 68m2 - 3.74 tỷ:
 gia_mua: 3.74 tỷ
 thu_nhap_thang_uoc_tinh: 12,240,000 VNĐ
 ti_suat_loi_nhuan: 3.93%/năm


In [31]:
# TAVILY search - remote server
instructions = """
Bạn là AI nghiên cứu BĐS.

Quy trình:
1. Dùng tavily_search để lấy overview nhanh
2. Nếu cần phân tích sâu → dùng tavily_research
3. Tóm tắt 3 insight

Ưu tiên tốc độ, chỉ dùng research khi thật sự cần
"""
async with MCPServerStdio(params=TAVILY_PARAMS, client_session_timeout_seconds=60) as mcp_tavily:
    tavily_tools = await mcp_tavily.list_tools()
    print(f"Tavily tools: {[t.name for t in tavily_tools]}")
    print()

    agent_search = Agent(
        name='tin-tuc-bds',
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[mcp_tavily]
    )

    with trace("tavily-search-bds"):
        r = await Runner.run(
            agent_search,
            f"Dùng tavily_research để nghiên cứu sâu về: 'xu hướng đầu tư bất động sản "
            f"tháng {datetime.now().strftime('%m/%Y')}. "
            f"Tóm tắt 3 điểm nổi bật nhất.",
            max_turns=5
        )
    print(r.final_output)

Tavily tools: ['tavily_search', 'tavily_extract', 'tavily_crawl', 'tavily_map', 'tavily_research']

Dựa trên thông tin tìm kiếm, dưới đây là ba điểm nổi bật về xu hướng đầu tư bất động sản tháng 04/2026:

1. **Tăng trưởng và chuyển dịch đầu tư**:
   - Thị trường bất động sản Việt Nam vào năm 2026 dự kiến sẽ chịu tác động từ lạm phát và lãi suất tăng, nhưng vẫn được thúc đẩy bởi dòng vốn FDI, đầu tư hạ tầng và các dự án lớn. Việc chuyển dịch dòng tiền sang các khu vực ven đô để tận dụng giá thuê cao hơn đang gia tăng.

2. **Nhu cầu đối với bất động sản nghỉ dưỡng và cho thuê ngắn hạn**:
   - Nhu cầu mua bất động sản để ở kết hợp với đầu tư cho thuê ngắn hạn đang tăng cao, đặc biệt ở những khu vực phát triển du lịch. Các nhà đầu tư trẻ và đối tượng nước ngoài đang tìm kiếm các cơ hội này.

3. **Thách thức pháp lý và cạnh tranh**:
   - Mặc dù có nhiều cơ hội đầu tư hấp dẫn, thị trường cũng đối diện với những thách thức về pháp lý, nguồn cung bất động sản hạn chế và sự cạnh tranh ngày càng

# Project: Agent Phân Tích BĐS đầu tư

In [32]:
instructions_bds = """
Bạn là chuyên gia phân tích bất động sản đầu tư tại Việt Nam với 10 năm kinh nghiệm.

Phương pháp làm việc:
1. Dùng memory tools để lưu lại mọi insight quan trọng khi phân tích
2. Dùng dữ liệu thị trường thực (ty gia, BĐS) để có số liệu chính xác
3. Dùng Tavily Search để bổ sung thông tin xu hướng thực tế mới nhất
4. Mọi khuyến nghị phải có số liệu cụ thể để support

Khi viết báo cáo:
- Đọc lại memory để đảm bảo không bỏ sót insight đã phân tích
- Format số tiền theo chuẩn Việt Nam (tỷ, triệu, ngàn đồng)
- Báo cáo phải actionable — khách hàng đọc xong biết phải làm gì ngay
- Ngôn ngữ: tiếng Việt hoàn toàn
"""

request = f"""
Khách hàng: Chị Thanh Tâm, 30 tuổi, ngân sách 4 tỷ VNĐ
Mục tiêu: Mua chung cư tại HCM hoặc Hà Nội để đầu tư cho thuê dài hạn

Hãy thực hiện phân tích đầu tư toàn diện qua 5 bước:

BƯỚC 1 — Bối cảnh tài chính vĩ mô
Lấy tỷ giá USD hôm nay.
Lưu vào memory entity "boi_canh_vi_mo" với observations về tình hình.

BƯỚC 2 — Khảo sát thị trường BĐS
Tìm tất cả chung cư dưới 4 tỷ tại HCM và tại Hà Nội.
Lấy thống kê giá trung bình tất cả khu vực.
Lưu các khu vực tiềm năng vào memory.

BƯỚC 3 — Tính ROI top 3 khu vực
Chọn 3 khu vực tốt nhất và tính gross yield cho căn chung cư điển hình.
Lưu kết quả ROI vào memory để so sánh.

BƯỚC 4 — Nghiên cứu xu hướng thị trường
Tìm kiếm tin tức gần nhất về: "đầu tư chung cư HCM Hà Nội 2026 cho thuê"
Lưu 2-3 insight từ tin tức vào memory.

BƯỚC 5 — Báo cáo khuyến nghị
Đọc lại toàn bộ memory, rồi viết báo cáo đầu tư hoàn chỉnh:

---
# BÁO CÁO PHÂN TÍCH ĐẦU TƯ BĐS CHO THUÊ
**Khách hàng:** Chị Thanh Tâm | **Ngân sách:** 4 tỷ VNĐ | **Ngày:** {datetime.now().strftime('%d/%m/%Y')}

## 1. Bối Cảnh Tài Chính
[Tỷ giá, ý nghĩa với nhà đầu tư]

## 2. Thị Trường Mục Tiêu
[So sánh HCM vs Hà Nội, mặt bằng giá, khu vực phù hợp ngân sách]

## 3. Phân Tích Hiệu Quả Đầu Tư

| Khu vực | Giá TB/m² | Diện tích | Giá mua | Yield | Hoàn vốn |
|---------|-----------|-----------|---------|-------|-----------|

## 4. Xu Hướng Thị Trường
[3 điểm nổi bật từ tin tức gần nhất]

## 5. Khuyến Nghị
**TOP PICK:** [1 khu vực cụ thể với lý do]
**NEXT BEST:** [Lựa chọn thay thế]
**LƯU Ý RỦI RO:** [2-3 rủi ro cần xem xét]
---

Sau khi xong, xác nhận đã lưu bao nhiêu entities vào memory.
"""

In [33]:
# Ket hop 3 loai server
async with MCPServerStdio(params=MEMORY_PARAMS, client_session_timeout_seconds=15) as mcp_mem:
    async with MCPServerStdio(params=DULIEU_VN_PARAMS, client_session_timeout_seconds=30) as mcp_vn:
        async with MCPServerStdio(params=TAVILY_PARAMS, client_session_timeout_seconds=60) as mcp_tavily:
            agent_bds = Agent(
                name="chuyen-gia-phan-tich-bds",
                instructions=instructions_bds,
                model="gpt-4o-mini",
                mcp_servers=[mcp_mem, mcp_vn, mcp_tavily]
            )
            with trace("phan-tich-bds-dau-tu-hcm-hn"):
                result = await Runner.run(agent_bds, request, max_turns=40)
                print(result.final_output)

### Đã hoàn thành lưu trữ các entities vào memory với tổng số lượng như sau:
- **Bối cảnh tài chính vĩ mô:** 1 entity
- **Khu vực BĐS HCM:** 1 entity
- **Khu vực BĐS HN:** 1 entity
- **ROI HCM:** 1 entity
- **ROI HN:** 1 entity
- **Xu hướng thị trường:** 1 entity

Tổng cộng có **6 entities** đã được lưu.

### Dưới đây là báo cáo phân tích đầu tư hoàn chỉnh:

---

# BÁO CÁO PHÂN TÍCH ĐẦU TƯ BĐS CHO THUÊ
**Khách hàng:** Chị Thanh Tâm | **Ngân sách:** 4 tỷ VNĐ | **Ngày:** 08/04/2026

## 1. Bối Cảnh Tài Chính
- **Tỷ giá USD hôm nay:**
  - Mua TM: 26,111 VNĐ
  - Mua CK: 26,141 VNĐ
  - Bán: 26,361 VNĐ (Vietcombank)

Tình hình tài chính vĩ mô hiện tại ở mức ổn định, nhưng cần theo dõi diễn biến thị trường bất động sản và chính sách thuế sắp tới.

## 2. Thị Trường Mục Tiêu
**So sánh HCM vs Hà Nội:**
- Giá trung bình chung cư:
  - HCM: 55 triệu VNĐ/m²
  - HN: 55 triệu VNĐ/m²

### Khu vực phù hợp ngân sách:
- **HCM:** 
  - Quận 7: Chung cư 68m², giá 3.74 tỷ VNĐ, gần Crescent Mall.
- **HN:** 
  -